In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys

import pandas as pd

direktori_aktif = Path.cwd()
direktori_project = direktori_aktif.parent if direktori_aktif.name.lower() == "notebooks" else direktori_aktif

direktori_src = direktori_project / "src"
direktori_models = direktori_project / "models"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_examples = direktori_project / "examples"

for folder in [direktori_src, direktori_models, direktori_outputs, direktori_examples]:
    folder.mkdir(parents=True, exist_ok=True)

file_wajib = {
    "model_terbaik_v5": direktori_models / "model_terbaik_multi_dataset_v5.pkl",
    "model_xgb_v5": direktori_models / "model_xgb_multi_dataset_v5.pkl",
    "daftar_fitur_v5": direktori_outputs / "daftar_fitur_multi_dataset_v5.json",
    "threshold_v5": direktori_outputs / "threshold_model_terbaik_v5.json",
    "url_intelligence": direktori_src / "url_intelligence.py",
    "engine_v4": direktori_src / "phishrisk_engine_v4.py",
    "public_ti": direktori_src / "public_threat_intelligence.py",
    "file_analyzer": direktori_src / "file_static_analyzer.py",
}

validasi_awal = pd.DataFrame([
    {
        "nama_file": nama,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_mb": round(lokasi.stat().st_size / (1024 * 1024), 2) if lokasi.exists() else 0,
    }
    for nama, lokasi in file_wajib.items()
])

display(validasi_awal)

if not validasi_awal["tersedia"].all():
    raise FileNotFoundError("Ada file wajib yang belum tersedia. Cek tabel validasi di atas.")

print("Semua file wajib STEP 16 tersedia.")

,nama_file,lokasi,tersedia,ukuran_mb
0,model_terbaik_v5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,True,584.68
1,model_xgb_v5,C:\Users\ASUS\PHISHING\models\model_xgb_multi_...,True,1.89
2,daftar_fitur_v5,C:\Users\ASUS\PHISHING\reports\outputs\daftar_...,True,0.00
3,threshold_v5,C:\Users\ASUS\PHISHING\reports\outputs\thresho...,True,0.00
4,url_intelligence,C:\Users\ASUS\PHISHING\src\url_intelligence.py,True,0.02
5,engine_v4,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v4.py,True,0.00
6,public_ti,C:\Users\ASUS\PHISHING\src\public_threat_intel...,True,0.01
7,file_analyzer,C:\Users\ASUS\PHISHING\src\file_static_analyze...,True,0.02


Semua file wajib STEP 16 tersedia.


In [2]:
isi_engine_v5 = '\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nfrom difflib import SequenceMatcher\nfrom typing import Any, Dict, Iterable\nimport ipaddress\nimport json\nimport re\nimport sys\n\nimport joblib\nimport pandas as pd\n\n\nclass PhishRiskEngineV5:\n    def __init__(self, direktori_project: str | Path | None = None, prefer_model: str = "best"):\n        self.direktori_project = Path(direktori_project) if direktori_project else Path.cwd()\n        if self.direktori_project.name.lower() == "src":\n            self.direktori_project = self.direktori_project.parent\n\n        self.direktori_src = self.direktori_project / "src"\n        self.direktori_models = self.direktori_project / "models"\n        self.direktori_outputs = self.direktori_project / "reports" / "outputs"\n        self.direktori_intelligence = self.direktori_project / "data" / "intelligence"\n\n        if str(self.direktori_src) not in sys.path:\n            sys.path.insert(0, str(self.direktori_src))\n\n        self.prefer_model = prefer_model\n        self.daftar_fitur = self._load_json_list(self.direktori_outputs / "daftar_fitur_multi_dataset_v5.json")\n        self.threshold_info = self._load_threshold()\n        self.model_path, self.model = self._load_model()\n        self.base_engine = self._load_engine_v4()\n        self._load_intelligence_tables()\n\n    def _load_json_list(self, lokasi: Path):\n        with open(lokasi, "r", encoding="utf-8") as file:\n            data = json.load(file)\n        if isinstance(data, list):\n            return data\n        for key in ["features", "daftar_fitur", "feature_names"]:\n            if isinstance(data, dict) and isinstance(data.get(key), list):\n                return data[key]\n        raise ValueError("Format daftar fitur V5 tidak dikenali.")\n\n    def _load_threshold(self):\n        lokasi = self.direktori_outputs / "threshold_model_terbaik_v5.json"\n        if not lokasi.exists():\n            return {"threshold_terbaik": 0.5}\n        with open(lokasi, "r", encoding="utf-8") as file:\n            return json.load(file)\n\n    def _load_model(self):\n        if self.prefer_model == "xgb":\n            kandidat = ["model_xgb_multi_dataset_v5.pkl", "model_terbaik_multi_dataset_v5.pkl"]\n        elif self.prefer_model == "rf":\n            kandidat = ["model_rf_multi_dataset_v5.pkl", "model_terbaik_multi_dataset_v5.pkl"]\n        else:\n            kandidat = [\n                "model_terbaik_multi_dataset_v5.pkl",\n                "model_xgb_multi_dataset_v5.pkl",\n                "model_rf_multi_dataset_v5.pkl",\n            ]\n\n        for nama in kandidat:\n            lokasi = self.direktori_models / nama\n            if lokasi.exists():\n                return lokasi, joblib.load(lokasi)\n\n        raise FileNotFoundError("Model V5 tidak ditemukan.")\n\n    def _load_engine_v4(self):\n        try:\n            import phishrisk_engine_v4\n            try:\n                return phishrisk_engine_v4.PhishRiskEngineV4(direktori_project=self.direktori_project)\n            except TypeError:\n                return phishrisk_engine_v4.PhishRiskEngineV4(str(self.direktori_project))\n        except Exception:\n            return None\n\n    def _load_intelligence_tables(self):\n        self.official_domains = []\n        self.brand_keywords = []\n        self.brand_map = {}\n        self.suspicious_weights = {}\n\n        lokasi_official = self.direktori_intelligence / "official_domains_global.csv"\n        lokasi_brand = self.direktori_intelligence / "brand_keywords_global.csv"\n        lokasi_suspicious = self.direktori_intelligence / "suspicious_keywords_global.csv"\n\n        if lokasi_official.exists():\n            data = pd.read_csv(lokasi_official)\n            if "domain" in data.columns:\n                self.official_domains = data["domain"].dropna().astype(str).str.lower().str.strip().unique().tolist()\n\n        if lokasi_brand.exists():\n            data = pd.read_csv(lokasi_brand)\n            if "keyword" in data.columns:\n                self.brand_keywords = data["keyword"].dropna().astype(str).str.lower().str.strip().unique().tolist()\n            for _, row in data.iterrows():\n                keyword = str(row.get("keyword", "")).strip().lower()\n                brand = str(row.get("brand", keyword)).strip()\n                if keyword:\n                    self.brand_map[keyword] = brand\n\n        if lokasi_suspicious.exists():\n            data = pd.read_csv(lokasi_suspicious)\n            for _, row in data.iterrows():\n                keyword = str(row.get("keyword", "")).strip().lower()\n                try:\n                    bobot = int(row.get("bobot", 1))\n                except Exception:\n                    bobot = 1\n                if keyword:\n                    self.suspicious_weights[keyword] = bobot\n\n    def bersihkan_url(self, url: Any) -> str:\n        if url is None or pd.isna(url):\n            return ""\n        return re.sub(r"\\s+", "", str(url).strip().replace("\\x00", ""))\n\n    def normalisasi_url(self, url: Any) -> str:\n        url = self.bersihkan_url(url)\n        if not url:\n            return ""\n        if not re.match(r"^https?://", url, flags=re.I):\n            return "https://" + url\n        return url\n\n    def ambil_domain(self, url: Any) -> str:\n        url = self.bersihkan_url(url)\n        if not url:\n            return ""\n        try:\n            url_parse = url if re.match(r"^https?://", url, flags=re.I) else "http://" + url\n            parsed = urlparse(url_parse)\n            domain = parsed.netloc.lower()\n            domain = domain.split("@")[-1].split(":")[0]\n            return domain.replace("www.", "", 1)\n        except Exception:\n            return ""\n\n    def ambil_tld(self, domain: str) -> str:\n        bagian = str(domain).split(".")\n        return bagian[-1].lower() if len(bagian) >= 2 else ""\n\n    def cek_domain_ip(self, domain: str) -> int:\n        try:\n            ipaddress.ip_address(domain)\n            return 1\n        except Exception:\n            return 0\n\n    def hitung_subdomain(self, domain: str) -> int:\n        bagian = str(domain).split(".")\n        return max(0, len(bagian) - 2) if len(bagian) > 2 else 0\n\n    def cek_domain_resmi(self, domain: str) -> int:\n        domain = str(domain).lower().strip()\n        for official in self.official_domains:\n            if domain == official or domain.endswith("." + official):\n                return 1\n        return 0\n\n    def cek_brand(self, url: str, domain: str):\n        teks = f"{url} {domain}".lower()\n        for keyword in self.brand_keywords:\n            if keyword and keyword in teks:\n                return 1, self.brand_map.get(keyword, keyword)\n        return 0, ""\n\n    def cek_suspicious_keywords(self, url: str):\n        teks = str(url).lower()\n        ketemu = []\n        skor = 0\n        for keyword, bobot in self.suspicious_weights.items():\n            if keyword in teks:\n                ketemu.append(keyword)\n                skor += bobot\n        return len(ketemu), skor, ", ".join(ketemu)\n\n    def ubah_digit_mirip(self, teks: str) -> str:\n        return str(teks).translate(str.maketrans({"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t"}))\n\n    def cek_lookalike(self, domain: str, is_official: int):\n        if is_official:\n            return 0, 0.0, ""\n        domain_bersih = self.ubah_digit_mirip(re.sub(r"[^a-z0-9]", "", str(domain).lower()))\n        skor_terbaik = 0.0\n        brand_terbaik = ""\n        for keyword in self.brand_keywords:\n            keyword_bersih = re.sub(r"[^a-z0-9]", "", keyword.lower())\n            if len(keyword_bersih) <= 2:\n                continue\n            skor = SequenceMatcher(None, domain_bersih, keyword_bersih).ratio()\n            if keyword_bersih in domain_bersih and keyword_bersih != domain_bersih:\n                skor = max(skor, 1.0)\n            if skor > skor_terbaik:\n                skor_terbaik = skor\n                brand_terbaik = self.brand_map.get(keyword, keyword)\n        return int(skor_terbaik >= 0.82), round(float(skor_terbaik), 4), brand_terbaik\n\n    def ekstrak_fitur_satu_url(self, url: Any) -> Dict[str, Any]:\n        url = self.normalisasi_url(url)\n        domain = self.ambil_domain(url)\n        tld = self.ambil_tld(domain)\n\n        panjang_url = len(url)\n        jumlah_huruf = sum(karakter.isalpha() for karakter in url)\n        jumlah_digit = sum(karakter.isdigit() for karakter in url)\n        jumlah_spesial = sum(not karakter.isalnum() for karakter in url)\n\n        fitur = {\n            "URLLength": panjang_url,\n            "DomainLength": len(domain),\n            "TLDLength": len(tld),\n            "NoOfSubDomain": self.hitung_subdomain(domain),\n            "IsHTTPS": int(url.lower().startswith("https://")),\n            "IsDomainIP": self.cek_domain_ip(domain),\n            "NoOfLettersInURL": jumlah_huruf,\n            "NoOfDegitsInURL": jumlah_digit,\n            "NoOfDigitsInURL": jumlah_digit,\n            "NoOfOtherSpecialCharsInURL": jumlah_spesial,\n            "SpacialCharRatioInURL": jumlah_spesial / panjang_url if panjang_url else 0,\n            "SpecialCharRatioInURL": jumlah_spesial / panjang_url if panjang_url else 0,\n            "LetterRatioInURL": jumlah_huruf / panjang_url if panjang_url else 0,\n            "DegitRatioInURL": jumlah_digit / panjang_url if panjang_url else 0,\n            "DigitRatioInURL": jumlah_digit / panjang_url if panjang_url else 0,\n            "NoOfEqualsInURL": url.count("="),\n            "NoOfQMarkInURL": url.count("?"),\n            "NoOfAmpersandInURL": url.count("&"),\n            "NoOfSlashInURL": url.count("/"),\n            "NoOfDotInURL": url.count("."),\n            "NoOfDashInURL": url.count("-"),\n            "NoOfAtInURL": url.count("@"),\n        }\n\n        for kolom in self.daftar_fitur:\n            if kolom.startswith("TLD_"):\n                fitur[kolom] = 0\n\n        kolom_tld = f"TLD_{tld}"\n        if kolom_tld in fitur:\n            fitur[kolom_tld] = 1\n        elif "TLD_lainnya" in fitur:\n            fitur["TLD_lainnya"] = 1\n\n        is_official = self.cek_domain_resmi(domain)\n        brand_detected_flag, brand_detected = self.cek_brand(url, domain)\n        suspicious_count, suspicious_score, suspicious_keywords = self.cek_suspicious_keywords(url)\n        lookalike_flag, lookalike_score, lookalike_brand = self.cek_lookalike(domain, is_official)\n\n        fitur.update({\n            "is_official_domain": is_official,\n            "brand_keyword_detected": brand_detected_flag,\n            "brand_but_not_official": int(brand_detected_flag == 1 and is_official == 0),\n            "suspicious_keyword_count": suspicious_count,\n            "suspicious_keyword_score": suspicious_score,\n            "lookalike_brand_detected": lookalike_flag,\n            "lookalike_score": lookalike_score,\n            "uses_punycode": int("xn--" in domain),\n            "uses_digit_substitution": int(any(karakter.isdigit() for karakter in domain)),\n            "hyphen_count": domain.count("-"),\n        })\n\n        fitur_final = {nama: fitur.get(nama, 0) for nama in self.daftar_fitur}\n        fitur_final["_url"] = url\n        fitur_final["_domain"] = domain\n        fitur_final["_brand_detected"] = brand_detected\n        fitur_final["_suspicious_keywords"] = suspicious_keywords\n        fitur_final["_lookalike_brand"] = lookalike_brand\n        return fitur_final\n\n    def prediksi_model_v5(self, url: Any) -> Dict[str, Any]:\n        fitur = self.ekstrak_fitur_satu_url(url)\n        fitur_model = {nama: fitur.get(nama, 0) for nama in self.daftar_fitur}\n        data_input = pd.DataFrame([fitur_model], columns=self.daftar_fitur)\n        if hasattr(self.model, "predict_proba"):\n            probabilitas = float(self.model.predict_proba(data_input)[0][1])\n        else:\n            probabilitas = float(self.model.predict(data_input)[0])\n        return {\n            "url": fitur["_url"],\n            "domain": fitur["_domain"],\n            "probabilitas_berisiko_v5": probabilitas,\n            "skor_model_v5": round(probabilitas * 100, 2),\n            "label_model_v5": "Berisiko" if probabilitas >= 0.5 else "Aman",\n            "brand_detected_v5": fitur["_brand_detected"],\n            "suspicious_keywords_v5": fitur["_suspicious_keywords"],\n            "lookalike_brand_v5": fitur["_lookalike_brand"],\n            **fitur_model,\n        }\n\n    def analisis_base(self, url: str) -> Dict[str, Any]:\n        if self.base_engine is None:\n            return {"url": url, "domain": self.ambil_domain(url), "intelligence_status": "base_engine_tidak_tersedia"}\n        try:\n            hasil = self.base_engine.analisis_url(url)\n            if isinstance(hasil, pd.Series):\n                return hasil.to_dict()\n            if isinstance(hasil, dict):\n                return hasil\n            return dict(hasil)\n        except Exception as error:\n            return {\n                "url": url,\n                "domain": self.ambil_domain(url),\n                "intelligence_status": "base_engine_gagal",\n                "base_error": str(error)[:300],\n            }\n\n    def kategori_dari_skor(self, skor: float):\n        if skor < 30:\n            return "Rendah", "Terlihat Aman"\n        if skor < 60:\n            return "Sedang", "Perlu Tinjauan"\n        if skor < 80:\n            return "Tinggi", "Berisiko"\n        return "Sangat Tinggi", "Berisiko"\n\n    def kalibrasi_final(self, hasil: Dict[str, Any]) -> Dict[str, Any]:\n        skor_model = float(hasil.get("skor_model_v5", 0))\n        skor_final = skor_model\n        alasan = []\n\n        intelligence_status = str(hasil.get("intelligence_status", "")).lower()\n        public_ti_status = str(hasil.get("public_ti_status", "")).lower()\n\n        is_official = int(float(hasil.get("is_official_domain", 0) or 0))\n        brand_but_not_official = int(float(hasil.get("brand_but_not_official", 0) or 0))\n        suspicious_score = int(float(hasil.get("suspicious_keyword_score", 0) or 0))\n        lookalike_detected = int(float(hasil.get("lookalike_brand_detected", 0) or 0))\n        uses_punycode = int(float(hasil.get("uses_punycode", 0) or 0))\n\n        if "terindikasi" in public_ti_status or "ancaman" in public_ti_status:\n            skor_final = max(skor_final, 95)\n            alasan.append("Public Threat Intelligence menemukan sinyal ancaman kuat.")\n        elif "perlu_tinjauan" in public_ti_status:\n            skor_final = max(skor_final, 60)\n            alasan.append("Public Threat Intelligence memberi sinyal perlu tinjauan.")\n        elif "catatan" in public_ti_status:\n            alasan.append("Public Threat Intelligence memberi catatan ringan.")\n\n        if "tiruan_brand_berisiko" in intelligence_status:\n            skor_final = max(skor_final, 92)\n            alasan.append("URL memakai nama brand tetapi bukan domain resmi.")\n        elif "domain_mirip_brand_berisiko" in intelligence_status:\n            skor_final = max(skor_final, 90)\n            alasan.append("Domain terlihat mirip dengan brand resmi dan memiliki sinyal tambahan.")\n        elif "domain_mirip_brand" in intelligence_status:\n            skor_final = max(skor_final, 82)\n            alasan.append("Domain terlihat mirip dengan brand resmi.")\n        elif "kata_mencurigakan_tinggi" in intelligence_status:\n            skor_final = max(skor_final, 72)\n            alasan.append("URL mengandung kata yang sering muncul pada serangan phishing.")\n\n        if brand_but_not_official:\n            skor_final = max(skor_final, 82)\n            alasan.append("Brand terdeteksi tetapi domain tidak cocok dengan daftar resmi.")\n\n        if suspicious_score >= 6:\n            skor_final = max(skor_final, 72)\n            alasan.append("Skor kata mencurigakan tinggi.")\n        elif suspicious_score >= 3:\n            skor_final = max(skor_final, 45)\n            alasan.append("Ada kata yang perlu diwaspadai.")\n\n        if lookalike_detected:\n            skor_final = max(skor_final, 78)\n            alasan.append("Domain memiliki pola mirip brand.")\n\n        if uses_punycode:\n            skor_final = max(skor_final, 85)\n            alasan.append("Domain memakai punycode yang perlu diperiksa manual.")\n\n        if is_official and "resmi_terlihat_aman" in intelligence_status:\n            if suspicious_score == 0 and not lookalike_detected and not uses_punycode:\n                skor_final = min(skor_final, 24)\n                alasan.append("Domain cocok dengan daftar resmi dan tidak punya sinyal mencurigakan kuat.")\n            else:\n                skor_final = min(max(skor_final, 35), 59)\n                alasan.append("Domain resmi tetapi tetap memiliki sinyal yang perlu ditinjau.")\n\n        skor_final = round(max(0, min(100, float(skor_final))), 2)\n        kategori, hasil_akhir = self.kategori_dari_skor(skor_final)\n\n        if hasil_akhir == "Terlihat Aman":\n            rekomendasi = "Alamat terlihat rendah risiko. Tetap pastikan sumber link tepercaya dan buka dari kanal resmi."\n        elif hasil_akhir == "Perlu Tinjauan":\n            rekomendasi = "Jangan langsung login atau memasukkan data pribadi. Cek domain resmi dan sumber link terlebih dahulu."\n        else:\n            rekomendasi = "Alamat berisiko. Jangan login, jangan isi data pribadi, jangan unduh file, dan laporkan jika perlu."\n\n        return {\n            "skor_final_v5": skor_final,\n            "kategori_risiko_v5": kategori,\n            "hasil_akhir_v5": hasil_akhir,\n            "alasan_v5": " ".join(alasan) if alasan else "Keputusan berdasarkan skor model V5 dan sinyal intelligence.",\n            "rekomendasi_v5": rekomendasi,\n        }\n\n    def analisis_url(self, url: Any) -> Dict[str, Any]:\n        url_normal = self.normalisasi_url(url)\n        hasil = {}\n        hasil.update(self.analisis_base(url_normal))\n        hasil.update(self.prediksi_model_v5(url_normal))\n        hasil.update(self.kalibrasi_final(hasil))\n        hasil["engine_version"] = "V5"\n        hasil["model_path_v5"] = str(self.model_path)\n        hasil["threshold_referensi_v5"] = self.threshold_info.get("threshold_terbaik", 0.5)\n        return hasil\n\n    def analisis_banyak_url(self, daftar_url: Iterable[Any]) -> pd.DataFrame:\n        return pd.DataFrame([self.analisis_url(url) for url in daftar_url if str(url).strip()])\n\n    def _fallback_hasil_file(self, lokasi_file: str | Path, pesan: str) -> Dict[str, Any]:\n        lokasi_file = Path(lokasi_file)\n\n        return {\n            "nama_file": lokasi_file.name,\n            "ekstensi": lokasi_file.suffix.lower(),\n            "engine_version": "V5",\n            "hasil_akhir_file_v5": "Perlu Tinjauan",\n            "kategori_final_file_v5": "Sedang",\n            "skor_final_file_v5": 50,\n            "rekomendasi_final_file_v5": pesan,\n        }\n\n    def _dataframe_file_ke_dict(self, data: pd.DataFrame, lokasi_file: str | Path) -> Dict[str, Any]:\n        if data.empty:\n            return self._fallback_hasil_file(lokasi_file, "Hasil file analyzer kosong. Periksa file secara manual.")\n\n        nama_file = Path(lokasi_file).name\n        data_pilih = data.copy()\n\n        if "nama_file" in data_pilih.columns:\n            cocok = data_pilih[data_pilih["nama_file"].astype(str) == nama_file]\n            if not cocok.empty:\n                data_pilih = cocok\n\n        return data_pilih.iloc[0].to_dict()\n\n    def _hasil_file_ke_dict(self, hasil: Any, lokasi_file: str | Path) -> Dict[str, Any]:\n        if hasil is None:\n            return self._fallback_hasil_file(lokasi_file, "File analyzer tidak mengembalikan hasil.")\n\n        if isinstance(hasil, dict):\n            return dict(hasil)\n\n        if isinstance(hasil, pd.Series):\n            return hasil.to_dict()\n\n        if isinstance(hasil, pd.DataFrame):\n            return self._dataframe_file_ke_dict(hasil, lokasi_file)\n\n        if isinstance(hasil, (list, tuple)):\n            if len(hasil) == 0:\n                return self._fallback_hasil_file(lokasi_file, "File analyzer mengembalikan list kosong.")\n\n            if all(isinstance(item, dict) for item in hasil):\n                return self._dataframe_file_ke_dict(pd.DataFrame(hasil), lokasi_file)\n\n            for item in hasil:\n                if isinstance(item, (dict, pd.Series, pd.DataFrame)):\n                    return self._hasil_file_ke_dict(item, lokasi_file)\n\n            return self._fallback_hasil_file(\n                lokasi_file,\n                "Format hasil file analyzer tidak dikenali. Periksa file secara manual.",\n            )\n\n        try:\n            return dict(hasil)\n        except Exception:\n            return self._fallback_hasil_file(\n                lokasi_file,\n                f"Format hasil file analyzer tidak bisa dikonversi: {type(hasil).__name__}.",\n            )\n\n    def _selaraskan_kolom_file_v5(self, hasil: Dict[str, Any]) -> Dict[str, Any]:\n        hasil = dict(hasil)\n\n        pasangan_kolom = {\n            "hasil_akhir_file_v3": "hasil_akhir_file_v5",\n            "kategori_final_file_v3": "kategori_final_file_v5",\n            "skor_final_file_v3": "skor_final_file_v5",\n            "rekomendasi_final_file_v3": "rekomendasi_final_file_v5",\n            "jumlah_url_berisiko_v3": "jumlah_url_berisiko_v5",\n            "jumlah_url_perlu_tinjauan_v3": "jumlah_url_perlu_tinjauan_v5",\n        }\n\n        for kolom_lama, kolom_baru in pasangan_kolom.items():\n            if kolom_lama in hasil and kolom_baru not in hasil:\n                hasil[kolom_baru] = hasil[kolom_lama]\n\n        if "hasil_akhir_file" in hasil and "hasil_akhir_file_v5" not in hasil:\n            hasil["hasil_akhir_file_v5"] = hasil["hasil_akhir_file"]\n\n        if "kategori_final_file" in hasil and "kategori_final_file_v5" not in hasil:\n            hasil["kategori_final_file_v5"] = hasil["kategori_final_file"]\n\n        if "rekomendasi_file" in hasil and "rekomendasi_final_file_v5" not in hasil:\n            hasil["rekomendasi_final_file_v5"] = hasil["rekomendasi_file"]\n\n        hasil.setdefault("hasil_akhir_file_v5", hasil.get("hasil_akhir_file_v3", "Perlu Tinjauan"))\n        hasil.setdefault("kategori_final_file_v5", hasil.get("kategori_final_file_v3", "Sedang"))\n        hasil.setdefault("rekomendasi_final_file_v5", "Periksa sumber file dan jangan menjalankan file mencurigakan di perangkat utama.")\n\n        return hasil\n\n    def analisis_file(self, lokasi_file: str | Path) -> Dict[str, Any]:\n        lokasi_file = Path(lokasi_file)\n\n        if self.base_engine is None or not hasattr(self.base_engine, "analisis_file"):\n            return self._fallback_hasil_file(\n                lokasi_file,\n                "File analyzer base tidak tersedia. Periksa file secara manual.",\n            )\n\n        try:\n            hasil_base = self.base_engine.analisis_file(lokasi_file)\n            hasil = self._hasil_file_ke_dict(hasil_base, lokasi_file)\n        except Exception as error:\n            hasil = self._fallback_hasil_file(\n                lokasi_file,\n                f"File analyzer gagal dijalankan: {str(error)[:250]}",\n            )\n\n        hasil.setdefault("nama_file", lokasi_file.name)\n        hasil.setdefault("ekstensi", lokasi_file.suffix.lower())\n        hasil = self._selaraskan_kolom_file_v5(hasil)\n        hasil["engine_version"] = "V5"\n        hasil["catatan_v5"] = "Analisis file memakai static analyzer yang sudah ada. Engine V5 dipakai untuk kalibrasi URL secara terpisah."\n\n        return hasil\n\n    def analisis_banyak_file(self, daftar_file: Iterable[str | Path]) -> pd.DataFrame:\n        hasil = []\n\n        for file in daftar_file:\n            try:\n                hasil.append(self.analisis_file(file))\n            except Exception as error:\n                hasil.append(\n                    self._fallback_hasil_file(\n                        file,\n                        f"Analisis file gagal: {str(error)[:250]}",\n                    )\n                )\n\n        return pd.DataFrame(hasil)\n\n    def analisis_url_di_dalam_file(self, lokasi_file: str | Path) -> pd.DataFrame:\n        if self.base_engine is None or not hasattr(self.base_engine, "analisis_url_di_dalam_file"):\n            return pd.DataFrame()\n\n        try:\n            data_url = self.base_engine.analisis_url_di_dalam_file(lokasi_file)\n\n            if isinstance(data_url, list):\n                data_url = pd.DataFrame(data_url)\n\n            if not isinstance(data_url, pd.DataFrame) or data_url.empty or "url" not in data_url.columns:\n                return pd.DataFrame()\n\n            hasil = []\n            for url in data_url["url"].dropna().astype(str).tolist():\n                item = self.analisis_url(url)\n                item["nama_file_sumber"] = Path(lokasi_file).name\n                hasil.append(item)\n\n            return pd.DataFrame(hasil)\n        except Exception:\n            return pd.DataFrame()\n'

lokasi_engine_v5 = direktori_src / "phishrisk_engine_v5.py"
lokasi_engine_v5.write_text(isi_engine_v5, encoding="utf-8")

print("Engine V5 berhasil dibuat:")
print(lokasi_engine_v5)


Engine V5 berhasil dibuat:
C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py


In [3]:
isi_cli_v5 = '\nfrom pathlib import Path\nimport argparse\nimport sys\n\nimport pandas as pd\n\nPROJECT_DIR = Path(__file__).resolve().parents[1]\nSRC_DIR = PROJECT_DIR / "src"\n\nif str(SRC_DIR) not in sys.path:\n    sys.path.insert(0, str(SRC_DIR))\n\nfrom phishrisk_engine_v5 import PhishRiskEngineV5\n\n\ndef simpan_output(data, lokasi_output):\n    lokasi_output = Path(lokasi_output)\n    lokasi_output.parent.mkdir(parents=True, exist_ok=True)\n\n    if isinstance(data, pd.DataFrame):\n        data.to_csv(lokasi_output, index=False, encoding="utf-8")\n    else:\n        pd.DataFrame([data]).to_csv(lokasi_output, index=False, encoding="utf-8")\n\n    return lokasi_output\n\n\ndef baca_url_dari_input(input_value, url_column="url"):\n    lokasi = Path(input_value)\n\n    if lokasi.exists() and lokasi.suffix.lower() == ".csv":\n        data = pd.read_csv(lokasi)\n\n        if url_column not in data.columns:\n            raise ValueError(f"Kolom URL tidak ditemukan: {url_column}")\n\n        return data[url_column].dropna().astype(str).tolist()\n\n    if lokasi.exists() and lokasi.suffix.lower() in [".txt", ".log"]:\n        return [\n            baris.strip()\n            for baris in lokasi.read_text(encoding="utf-8", errors="ignore").splitlines()\n            if baris.strip()\n        ]\n\n    return [input_value]\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="PhishRisk Engine V5 CLI")\n    parser.add_argument("--mode", choices=["url", "urls", "file", "folder"], required=True)\n    parser.add_argument("--input", required=True)\n    parser.add_argument("--output", default="")\n    parser.add_argument("--url-column", default="url")\n    parser.add_argument("--model-mode", choices=["best", "rf", "xgb"], default="best")\n\n    args = parser.parse_args()\n    engine = PhishRiskEngineV5(direktori_project=PROJECT_DIR, prefer_model=args.model_mode)\n\n    if args.mode == "url":\n        hasil = pd.DataFrame([engine.analisis_url(args.input)])\n        output_default = PROJECT_DIR / "reports" / "outputs" / "hasil_cli_url_engine_v5.csv"\n    elif args.mode == "urls":\n        hasil = engine.analisis_banyak_url(baca_url_dari_input(args.input, args.url_column))\n        output_default = PROJECT_DIR / "reports" / "outputs" / "hasil_cli_banyak_url_engine_v5.csv"\n    elif args.mode == "file":\n        hasil = pd.DataFrame([engine.analisis_file(args.input)])\n        output_default = PROJECT_DIR / "reports" / "outputs" / "hasil_cli_file_engine_v5.csv"\n    else:\n        folder = Path(args.input)\n        if not folder.exists():\n            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")\n        hasil = engine.analisis_banyak_file([item for item in folder.iterdir() if item.is_file()])\n        output_default = PROJECT_DIR / "reports" / "outputs" / "hasil_cli_folder_engine_v5.csv"\n\n    lokasi_output = Path(args.output) if args.output else output_default\n    simpan_output(hasil, lokasi_output)\n\n    print("PhishRisk Engine V5 selesai.")\n    print("Mode:", args.mode)\n    print("Output:", lokasi_output)\n\n    kolom_ringkas = [kolom for kolom in ["hasil_akhir_v5", "kategori_risiko_v5", "skor_final_v5", "engine_version"] if kolom in hasil.columns]\n    if kolom_ringkas:\n        print(hasil[kolom_ringkas].head(10).to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n'

lokasi_cli_v5 = direktori_src / "run_phishrisk_v5.py"
lokasi_cli_v5.write_text(isi_cli_v5, encoding="utf-8")

print("CLI V5 berhasil dibuat:")
print(lokasi_cli_v5)

CLI V5 berhasil dibuat:
C:\Users\ASUS\PHISHING\src\run_phishrisk_v5.py


In [4]:
import sys
import importlib

if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import phishrisk_engine_v5
importlib.reload(phishrisk_engine_v5)

engine_v5 = phishrisk_engine_v5.PhishRiskEngineV5(
    direktori_project=direktori_project,
    prefer_model="best",
)

url_uji_v5 = [
    "https://praktikum.gunadarma.ac.id",
    "https://baak.gunadarma.ac.id",
    "https://www.bca.co.id",
    "https://www.shopee.co.id",
    "https://www.microsoft.com",
    "http://rricrosoft.com",
    "http://rnicrosoft.com",
    "http://micros0ft-login-update.test",
    "http://bca-login-update.test",
    "http://paypal-verify-account.test",
    "http://praktikum-gunadarma-login-update.test",
    "https://xn--micrsoft-q4a.test",
    "http://155.94.163.206/ai/?authenticated=true&account=login",
]

hasil_uji_url_v5 = engine_v5.analisis_banyak_url(url_uji_v5)
lokasi_hasil_uji_url_v5 = direktori_outputs / "hasil_uji_engine_v5_url.csv"
hasil_uji_url_v5.to_csv(lokasi_hasil_uji_url_v5, index=False, encoding="utf-8")

kolom_tampil = [
    "url", "domain", "skor_model_v5", "skor_final_v5",
    "kategori_risiko_v5", "hasil_akhir_v5",
    "intelligence_status", "public_ti_status",
    "alasan_v5", "rekomendasi_v5",
]
kolom_tampil = [kolom for kolom in kolom_tampil if kolom in hasil_uji_url_v5.columns]

print("Uji URL Engine V5 selesai.")
print(lokasi_hasil_uji_url_v5)
display(hasil_uji_url_v5[kolom_tampil])

Uji URL Engine V5 selesai.
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v5_url.csv


,url,domain,skor_model_v5,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,intelligence_status,public_ti_status,alasan_v5,rekomendasi_v5
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,68.49,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...
1,https://baak.gunadarma.ac.id,baak.gunadarma.ac.id,32.91,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...
2,https://www.bca.co.id,bca.co.id,7.99,7.99,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...
3,https://www.shopee.co.id,shopee.co.id,7.58,7.58,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...
4,https://www.microsoft.com,microsoft.com,3.63,3.63,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...
5,http://rricrosoft.com,rricrosoft.com,34.94,82.00,Sangat Tinggi,Berisiko,domain_mirip_brand,tidak_ditemukan_di_public_ti,Domain terlihat mirip dengan brand resmi.,"Alamat berisiko. Jangan login, jangan isi data..."
6,http://rnicrosoft.com,rnicrosoft.com,34.94,82.00,Sangat Tinggi,Berisiko,domain_mirip_brand,tidak_ditemukan_di_public_ti,Domain terlihat mirip dengan brand resmi.,"Alamat berisiko. Jangan login, jangan isi data..."
7,http://micros0ft-login-update.test,micros0ft-login-update.test,90.26,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,tidak_ditemukan_di_public_ti,URL memakai nama brand tetapi bukan domain res...,"Alamat berisiko. Jangan login, jangan isi data..."
8,http://bca-login-update.test,bca-login-update.test,91.91,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,tidak_ditemukan_di_public_ti,URL memakai nama brand tetapi bukan domain res...,"Alamat berisiko. Jangan login, jangan isi data..."
9,http://paypal-verify-account.test,paypal-verify-account.test,87.32,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,tidak_ditemukan_di_public_ti,URL memakai nama brand tetapi bukan domain res...,"Alamat berisiko. Jangan login, jangan isi data..."


In [5]:
folder_sample_file = direktori_project / "data" / "samples_metadata" / "file_samples"

if folder_sample_file.exists():
    daftar_file_sample = [file for file in folder_sample_file.iterdir() if file.is_file()]

    hasil_file = []
    hasil_url_dalam_file = []

    for file in daftar_file_sample:
        hasil_file.append(engine_v5.analisis_file(file))

        try:
            data_url_file = engine_v5.analisis_url_di_dalam_file(file)
            if isinstance(data_url_file, pd.DataFrame) and not data_url_file.empty:
                hasil_url_dalam_file.append(data_url_file)
        except Exception:
            pass

    hasil_uji_file_v5 = pd.DataFrame(hasil_file)

    lokasi_hasil_uji_file_v5 = direktori_outputs / "hasil_uji_engine_v5_file.csv"
    hasil_uji_file_v5.to_csv(lokasi_hasil_uji_file_v5, index=False, encoding="utf-8")

    if hasil_url_dalam_file:
        hasil_uji_url_dalam_file_v5 = pd.concat(hasil_url_dalam_file, ignore_index=True)
    else:
        hasil_uji_url_dalam_file_v5 = pd.DataFrame()

    lokasi_hasil_uji_url_dalam_file_v5 = direktori_outputs / "hasil_uji_engine_v5_url_dalam_file.csv"
    hasil_uji_url_dalam_file_v5.to_csv(lokasi_hasil_uji_url_dalam_file_v5, index=False, encoding="utf-8")

    print("Uji file Engine V5 selesai.")
    print("Output file:", lokasi_hasil_uji_file_v5)
    print("Output URL dalam file:", lokasi_hasil_uji_url_dalam_file_v5)

    kolom_tampil_file = [
        "nama_file",
        "ekstensi",
        "hasil_akhir_file_v5",
        "kategori_final_file_v5",
        "skor_final_file_v5",
        "engine_version",
        "rekomendasi_final_file_v5",
    ]
    kolom_tampil_file = [kolom for kolom in kolom_tampil_file if kolom in hasil_uji_file_v5.columns]

    display(hasil_uji_file_v5[kolom_tampil_file].head(20))
else:
    lokasi_hasil_uji_file_v5 = direktori_outputs / "hasil_uji_engine_v5_file.csv"
    lokasi_hasil_uji_url_dalam_file_v5 = direktori_outputs / "hasil_uji_engine_v5_url_dalam_file.csv"
    pd.DataFrame().to_csv(lokasi_hasil_uji_file_v5, index=False, encoding="utf-8")
    pd.DataFrame().to_csv(lokasi_hasil_uji_url_dalam_file_v5, index=False, encoding="utf-8")

    print("Folder sample file belum tersedia:")
    print(folder_sample_file)


Uji file Engine V5 selesai.
Output file: C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v5_file.csv
Output URL dalam file: C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v5_url_dalam_file.csv


,nama_file,ekstensi,hasil_akhir_file_v5,kategori_final_file_v5,skor_final_file_v5,engine_version,rekomendasi_final_file_v5
0,contoh_aplikasi_dummy.apk,.apk,Berisiko,Sangat Tinggi,100,V5,File berisiko. Jangan dibuka atau dijalankan s...
1,contoh_arsip_mencurigakan.zip,.zip,Berisiko,Sangat Tinggi,89,V5,File berisiko. Jangan dibuka atau dijalankan s...
2,contoh_catatan_aman.txt,.txt,Terlihat Aman,Rendah,12,V5,File terlihat rendah risiko berdasarkan pemeri...
3,contoh_dokumen_link.docx,.docx,Berisiko,Sangat Tinggi,93,V5,File berisiko. Jangan dibuka atau dijalankan s...
4,contoh_halaman_login.html,.html,Berisiko,Sangat Tinggi,80,V5,File berisiko. Jangan dibuka atau dijalankan s...
5,contoh_pdf_link.pdf,.pdf,Berisiko,Sangat Tinggi,100,V5,File berisiko. Jangan dibuka atau dijalankan s...
6,contoh_pesan_mencurigakan.txt,.txt,Berisiko,Sangat Tinggi,80,V5,File berisiko. Jangan dibuka atau dijalankan s...


In [6]:
import phishrisk_engine_v4
importlib.reload(phishrisk_engine_v4)

try:
    engine_v4 = phishrisk_engine_v4.PhishRiskEngineV4(direktori_project=direktori_project)
except TypeError:
    engine_v4 = phishrisk_engine_v4.PhishRiskEngineV4(str(direktori_project))

data_perbandingan = []

for url in url_uji_v5:
    hasil_v4 = engine_v4.analisis_url(url)
    hasil_v5 = engine_v5.analisis_url(url)

    data_perbandingan.append({
        "url": url,
        "hasil_v4": hasil_v4.get("hasil_akhir_v4", hasil_v4.get("hasil_akhir", "")),
        "kategori_v4": hasil_v4.get("kategori_risiko_v4", hasil_v4.get("kategori_risiko", "")),
        "skor_v4": hasil_v4.get("skor_final_v4", hasil_v4.get("skor_final", "")),
        "hasil_v5": hasil_v5.get("hasil_akhir_v5", ""),
        "kategori_v5": hasil_v5.get("kategori_risiko_v5", ""),
        "skor_model_v5": hasil_v5.get("skor_model_v5", ""),
        "skor_final_v5": hasil_v5.get("skor_final_v5", ""),
        "intelligence_status": hasil_v5.get("intelligence_status", ""),
        "public_ti_status": hasil_v5.get("public_ti_status", ""),
    })

data_perbandingan_v4_v5 = pd.DataFrame(data_perbandingan)
lokasi_perbandingan_v4_v5 = direktori_outputs / "perbandingan_engine_v4_dan_v5.csv"
data_perbandingan_v4_v5.to_csv(lokasi_perbandingan_v4_v5, index=False, encoding="utf-8")

print("Perbandingan V4 dan V5 selesai.")
print(lokasi_perbandingan_v4_v5)
display(data_perbandingan_v4_v5)

Perbandingan V4 dan V5 selesai.
C:\Users\ASUS\PHISHING\reports\outputs\perbandingan_engine_v4_dan_v5.csv


,url,hasil_v4,kategori_v4,skor_v4,hasil_v5,kategori_v5,skor_model_v5,skor_final_v5,intelligence_status,public_ti_status
0,https://praktikum.gunadarma.ac.id,Terlihat Aman,Rendah,20.40,Terlihat Aman,Rendah,68.49,24.00,resmi_terlihat_aman,tidak_ditemukan_di_public_ti
1,https://baak.gunadarma.ac.id,Terlihat Aman,Rendah,24.00,Terlihat Aman,Rendah,32.91,24.00,resmi_terlihat_aman,tidak_ditemukan_di_public_ti
2,https://www.bca.co.id,Terlihat Aman,Rendah,4.05,Terlihat Aman,Rendah,7.99,7.99,resmi_terlihat_aman,tidak_ditemukan_di_public_ti
3,https://www.shopee.co.id,Terlihat Aman,Rendah,9.22,Terlihat Aman,Rendah,7.58,7.58,resmi_terlihat_aman,tidak_ditemukan_di_public_ti
4,https://www.microsoft.com,Terlihat Aman,Rendah,24.00,Terlihat Aman,Rendah,3.63,3.63,resmi_terlihat_aman,tidak_ditemukan_di_public_ti
5,http://rricrosoft.com,Berisiko,Sangat Tinggi,99.60,Berisiko,Sangat Tinggi,34.94,82.00,domain_mirip_brand,tidak_ditemukan_di_public_ti
6,http://rnicrosoft.com,Berisiko,Sangat Tinggi,99.60,Berisiko,Sangat Tinggi,34.94,82.00,domain_mirip_brand,tidak_ditemukan_di_public_ti
7,http://micros0ft-login-update.test,Berisiko,Sangat Tinggi,98.80,Berisiko,Sangat Tinggi,90.26,92.00,tiruan_brand_berisiko,tidak_ditemukan_di_public_ti
8,http://bca-login-update.test,Berisiko,Sangat Tinggi,99.60,Berisiko,Sangat Tinggi,91.91,92.00,tiruan_brand_berisiko,tidak_ditemukan_di_public_ti
9,http://paypal-verify-account.test,Berisiko,Sangat Tinggi,100.00,Berisiko,Sangat Tinggi,87.32,92.00,tiruan_brand_berisiko,tidak_ditemukan_di_public_ti


In [7]:
input_url_step16 = direktori_examples / "input_url_step16_engine_v5.csv"

pd.DataFrame({"url": url_uji_v5}).to_csv(
    input_url_step16,
    index=False,
    encoding="utf-8",
)

lokasi_cli_test_v5 = direktori_outputs / "hasil_test_cli_engine_v5.csv"

perintah = [
    sys.executable,
    str(direktori_src / "run_phishrisk_v5.py"),
    "--mode",
    "urls",
    "--input",
    str(input_url_step16),
    "--url-column",
    "url",
    "--output",
    str(lokasi_cli_test_v5),
    "--model-mode",
    "best",
]

hasil_cli = subprocess.run(perintah, capture_output=True, text=True)

print("STDOUT:")
print(hasil_cli.stdout)
print("STDERR:")
print(hasil_cli.stderr)
print("Return code:", hasil_cli.returncode)

if hasil_cli.returncode != 0:
    raise RuntimeError("CLI V5 gagal dijalankan.")

data_cli_v5 = pd.read_csv(lokasi_cli_test_v5)

print("Output CLI V5:")
print(lokasi_cli_test_v5)
display(data_cli_v5.head(20))

STDOUT:
PhishRisk Engine V5 selesai.
Mode: urls
Output: C:\Users\ASUS\PHISHING\reports\outputs\hasil_test_cli_engine_v5.csv
hasil_akhir_v5 kategori_risiko_v5  skor_final_v5 engine_version
 Terlihat Aman             Rendah          24.00             V5
 Terlihat Aman             Rendah          24.00             V5
 Terlihat Aman             Rendah           7.99             V5
 Terlihat Aman             Rendah           7.58             V5
 Terlihat Aman             Rendah           3.63             V5
      Berisiko      Sangat Tinggi          82.00             V5
      Berisiko      Sangat Tinggi          82.00             V5
      Berisiko      Sangat Tinggi          92.00             V5
      Berisiko      Sangat Tinggi          92.00             V5
      Berisiko      Sangat Tinggi          92.00             V5

STDERR:

Return code: 0
Output CLI V5:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_test_cli_engine_v5.csv


,url,domain,tld,probabilitas_model,skor_model,label_model,skor_final,kategori_risiko,hasil_akhir,rekomendasi,...,brand_keyword_detected,suspicious_keyword_count,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,alasan_v5,rekomendasi_v5,engine_version,model_path_v5,threshold_referensi_v5
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,id,0.204000,20.40,Legitimate,20.40,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,24.00,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
1,https://baak.gunadarma.ac.id,baak.gunadarma.ac.id,id,0.316000,31.60,Legitimate,24.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,24.00,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
2,https://www.bca.co.id,bca.co.id,id,0.040543,4.05,Legitimate,4.05,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,7.99,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
3,https://www.shopee.co.id,shopee.co.id,id,0.092183,9.22,Legitimate,9.22,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,7.58,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
4,https://www.microsoft.com,microsoft.com,com,0.480013,48.00,Legitimate,24.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,3.63,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
5,http://rricrosoft.com,rricrosoft.com,com,0.996000,99.60,Phishing,99.60,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,...,0,0,82.00,Sangat Tinggi,Berisiko,Domain terlihat mirip dengan brand resmi.,"Alamat berisiko. Jangan login, jangan isi data...",V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
6,http://rnicrosoft.com,rnicrosoft.com,com,0.996000,99.60,Phishing,99.60,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,...,0,0,82.00,Sangat Tinggi,Berisiko,Domain terlihat mirip dengan brand resmi.,"Alamat berisiko. Jangan login, jangan isi data...",V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
7,http://micros0ft-login-update.test,micros0ft-login-update.test,test,0.988000,98.80,Phishing,98.80,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,...,0,2,92.00,Sangat Tinggi,Berisiko,URL memakai nama brand tetapi bukan domain res...,"Alamat berisiko. Jangan login, jangan isi data...",V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
8,http://bca-login-update.test,bca-login-update.test,test,0.996000,99.60,Phishing,99.60,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,...,1,2,92.00,Sangat Tinggi,Berisiko,URL memakai nama brand tetapi bukan domain res...,"Alamat berisiko. Jangan login, jangan isi data...",V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
9,http://paypal-verify-account.test,paypal-verify-account.test,test,1.000000,100.00,Phishing,100.00,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,...,1,2,92.00,Sangat Tinggi,Berisiko,URL memakai nama brand tetapi bukan domain res...,"Alamat berisiko. Jangan login, jangan isi data...",V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15


In [8]:
file_output_step16 = [
    lokasi_engine_v5,
    lokasi_cli_v5,
    lokasi_hasil_uji_url_v5,
    lokasi_hasil_uji_file_v5 if "lokasi_hasil_uji_file_v5" in globals() else None,
    lokasi_hasil_uji_url_dalam_file_v5 if "lokasi_hasil_uji_url_dalam_file_v5" in globals() else None,
    lokasi_perbandingan_v4_v5,
    lokasi_cli_test_v5,
    input_url_step16,
]

data_validasi_step16 = []

for lokasi in file_output_step16:
    if lokasi is None:
        continue

    lokasi = Path(lokasi)
    data_validasi_step16.append({
        "nama_file": lokasi.name,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_kb": round(lokasi.stat().st_size / 1024, 2) if lokasi.exists() else 0,
    })

data_validasi_step16 = pd.DataFrame(data_validasi_step16)
lokasi_validasi_step16 = direktori_outputs / "validasi_step16_integrasi_engine_v5.csv"
data_validasi_step16.to_csv(lokasi_validasi_step16, index=False, encoding="utf-8")

metadata_step16 = {
    "nama_notebook": "16_integrasi_engine_v5.ipynb",
    "nama_tahap": "Integrasi Engine V5",
    "status": "selesai",
    "engine": "PhishRisk Engine V5",
    "model_digunakan": str(engine_v5.model_path),
    "fitur_digunakan": str(direktori_outputs / "daftar_fitur_multi_dataset_v5.json"),
    "threshold_referensi": engine_v5.threshold_info,
    "file_engine_v5": str(lokasi_engine_v5),
    "file_cli_v5": str(lokasi_cli_v5),
    "output_uji_url": str(lokasi_hasil_uji_url_v5),
    "output_perbandingan_v4_v5": str(lokasi_perbandingan_v4_v5),
    "validasi_step16": str(lokasi_validasi_step16),
    "catatan": [
        "Engine V5 memakai model Multi Dataset V5.",
        "Keputusan final memakai kalibrasi bertingkat, bukan threshold tunggal 0.15.",
        "Official domain checker tetap dipakai agar website resmi tidak langsung ditandai berisiko.",
        "Public Threat Intelligence dipakai sebagai sinyal tambahan melalui Engine V4.",
        "Mode deployment ringan dapat memakai model_xgb_multi_dataset_v5.pkl.",
    ],
    "tanggal_selesai": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

lokasi_metadata_step16 = direktori_outputs / "metadata_step16_integrasi_engine_v5.json"
lokasi_metadata_step16.write_text(
    json.dumps(metadata_step16, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

catatan_step16 = f"""
CATATAN FINAL STEP 16

Notebook:
16_integrasi_engine_v5.ipynb

Output utama:
{lokasi_engine_v5}

CLI:
{lokasi_cli_v5}

Model:
{engine_v5.model_path}

Catatan:
Engine V5 sudah menggabungkan model multi-dataset, URL Intelligence, Public Threat Intelligence, dan kalibrasi final.
Tahap berikutnya adalah validasi lanjutan Engine V5 sebelum website memakai engine baru.
"""

lokasi_catatan_step16 = direktori_outputs / "catatan_final_step16_integrasi_engine_v5.txt"
lokasi_catatan_step16.write_text(catatan_step16, encoding="utf-8")

print(catatan_step16)
print("Metadata:", lokasi_metadata_step16)
print("Validasi:", lokasi_validasi_step16)

display(data_validasi_step16)


CATATAN FINAL STEP 16

Notebook:
16_integrasi_engine_v5.ipynb

Output utama:
C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py

CLI:
C:\Users\ASUS\PHISHING\src\run_phishrisk_v5.py

Model:
C:\Users\ASUS\PHISHING\models\model_terbaik_multi_dataset_v5.pkl

Catatan:
Engine V5 sudah menggabungkan model multi-dataset, URL Intelligence, Public Threat Intelligence, dan kalibrasi final.
Tahap berikutnya adalah validasi lanjutan Engine V5 sebelum website memakai engine baru.

Metadata: C:\Users\ASUS\PHISHING\reports\outputs\metadata_step16_integrasi_engine_v5.json
Validasi: C:\Users\ASUS\PHISHING\reports\outputs\validasi_step16_integrasi_engine_v5.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,phishrisk_engine_v5.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py,True,24.69
1,run_phishrisk_v5.py,C:\Users\ASUS\PHISHING\src\run_phishrisk_v5.py,True,3.27
2,hasil_uji_engine_v5_url.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,19.76
3,hasil_uji_engine_v5_file.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,8.56
4,hasil_uji_engine_v5_url_dalam_file.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,0.00
5,perbandingan_engine_v4_dan_v5.csv,C:\Users\ASUS\PHISHING\reports\outputs\perband...,True,1.94
6,hasil_test_cli_engine_v5.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,20.60
7,input_url_step16_engine_v5.csv,C:\Users\ASUS\PHISHING\examples\input_url_step...,True,0.42
